# Interfaz para bases SQL usando SQLite3

Este *notebook* esta diseñado para ayudarte a lanzar una aplicación en la que puedes hacer *queries* a algunas bases SQL

## Instrucciones

1. Da *clic* en el botón **Conectar** en el extremo superior derecho y espera unos momentos para que se te asignen recursos
1. Ubica el botón **Ejecutar todo** y dale clic. Espera unos momentos mientras la aplicación se configura y ejecuta. Si lo deseas puedes ocultar todas las celdas intermedias
1. Ve al final del *notebook* en la sección **Iniciar Interfaz** y da clic en la `url` que se muestra que debe tener la forma : `https://xxxxxxxxxxxxxxxxx.gradio.live`, esto debe abrir una pestaña nueva 
1. Esperar que la aplicación cargue

> Si la aplicación deja de funcionar actualiza el notebook y repite los pasos


## Configuracion y código

### Cargando paquetes necesarios

In [ ]:
!pip install -q gradio || echo "Ya está instalada"

In [ ]:
import os
import sqlite3
import pandas as pd
import gradio as gr
import urllib.request

### Preparando bases de datos

In [ ]:
!git clone https://github.com/zyntonyson/dbs_sql.git

In [ ]:
DB_DIR='dbs_sql/dbs'
DBS = {f.replace('.db','').title():os.path.join(DB_DIR,f) for f in os.listdir(DB_DIR) if f.endswith(".db")}


### Preparando Interfaz


In [ ]:
def run_query(db_name, query):
  db_path=DBS[db_name]
  if not query.strip():
      return pd.DataFrame({"Error": ["Debes ingresar una consulta SQL."]})
  try:
      with sqlite3.connect(db_path) as conn:
          df = pd.read_sql_query(query, conn)
      return df
  except Exception as e:
      return pd.DataFrame({"Error": [str(e)]})

In [ ]:
def make_interface():
    with gr.Blocks(title="SQLite Query Runner") as demo:
        gr.Markdown("## 🧮 Consultas SQL")

        with gr.Row():
            # Lado izquierdo: selección + query + botón
            with gr.Column(scale=1):
                db_select = gr.Dropdown(
                    choices=list(DBS.keys()),
                    label="Base de datos",
                    value=list(DBS.keys())[1] #<- Sakila
                )

                query_input = gr.Textbox(
                    label="Ingresa tu query:",
                    value=""" 
                    
                    SELECT name
                    FROM sqlite_master
                    WHERE type = 'table'
                    
                    #Ver tablas disponibles
                     """,
                    
                    lines=10
                )

                boton = gr.Button("Run Query")

            # Lado derecho: resultado
            with gr.Column(scale=2):
                salida = gr.Dataframe(
                    label="Resultado",
                    wrap=True,# Ajusta texto en celdas
                    #interactive=True, # Hace que las columnas tengan ancho uniforme,
                    show_fullscreen_button=True,
                    show_copy_button=True,
                    min_width=200

                   )

        boton.click(fn=run_query, inputs=[db_select, query_input], outputs=salida)

    return demo



## Iniciar interfaz

In [ ]:
# --- Lanzar interfaz dentro del notebook ---
demo = make_interface()
demo.launch()  # inline=True → se muestra dentro del notebook